<a href="https://colab.research.google.com/github/saqib0-cpu/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: The gradient-boosted model achieved an F1 of 0.60 versus 0.154 for the
rule-based baseline under an 80/20 stratified split — described as "nearly a fourfold
improvement."

My methodology question: The dataset contains multiple content items per client
(client_hash_id repeats across rows). Under a random stratified split, items from the
same client can end up in both train and test. Since client-level patterns may be shared
across a client's pages, I would ask: how much of this F1 gain reflects genuine
generalizable signal versus the model learning client-specific patterns that happen to
repeat between train and test? A client-grouped split would help isolate this.


Finding 2: Recent click-through rate (ctr_b) accounts for 77.9% of the model's feature
importance, far ahead of position or click momentum. The paper itself notes this
dominance "deserves scrutiny" since ctr_b is partly derived from the same window as
some baseline signals.

My methodology question: The paper raises this caveat but does not resolve it with a
concrete check. I would ask: was a direct audit performed to confirm ctr_b does not
indirectly encode information correlated with the label window, beyond simply confirming
the windows are temporally separate? Temporal separation alone does not rule out
indirect leakage.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=features_df['client_hash_id']))

X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(features_df.iloc[train_idx]['client_hash_id'])
test_clients = set(features_df.iloc[test_idx]['client_hash_id'])
print("Client overlap between train/test:", len(train_clients & test_clients))

model_grouped = GradientBoostingClassifier(random_state=42)
model_grouped.fit(X_train_g, y_train_g)
preds_g = model_grouped.predict(X_test_g)

print(classification_report(y_test_g, preds_g))
grouped_f1 = f1_score(y_test_g, preds_g)

print(f"Before (random split, Week 5): F1 = 0.598")
print(f"After (client-grouped split): F1 = {grouped_f1:.3f}")

Observed: Under the random split (Week 5), the model reached F1 = 0.598. Under the
client-grouped split, the measured F1 was [insert actual number after running].
[If lower:] This suggests part of the original score was inflated by client-level
patterns shared across train and test. [If similar:] Performance held steady,
suggesting the model generalizes across clients rather than memorizing client-specific
behavior.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
leakage_notes = {
    'clicks_delta_pct': 'Built only from Windows A and B — before the label window. Safe.',
    'position_delta': 'Built only from Windows A and B. Safe.',
    'ctr_b': 'Built from Window B, which precedes the label window (Window C). No direct overlap.',
    'ctr_a': 'Built from Window A. Safe.',
    'search_volume': 'Static content metadata — need to confirm it was not updated after Window C.',
    'competition_level': 'Static metadata — same caveat as search_volume.',
    'word_count': 'Static snapshot — could reflect a content edit made in response to decline, which would leak the outcome.',
}
for f, note in leakage_notes.items():
    print(f"{f}: {note}")

import pandas as pd
features_df['content_updated_date'] = pd.to_datetime(features_df['content_updated_date'], errors='coerce')
updated_during_label_window = features_df[
    (features_df['content_updated_date'] >= '2026-02-01') &
    (features_df['content_updated_date'] <= '2026-04-30')
]
print(f"Pages updated during label window C: {len(updated_during_label_window)} of {len(features_df)}")

Observed: The behavioral features (clicks_delta_pct, position_delta, ctr_a, ctr_b) are
constructed strictly from Windows A and B, both of which precede the label window
(Window C) — no direct leakage found there. The static content features (word_count,
search_volume, competition_level) carry a directional risk: if content was edited during
or after the label window in response to a page already declining, these fields could
indirectly encode the outcome. The measured count of pages updated during Window C is
[insert number from output] out of [insert total] — [small/large] enough that this risk
is [likely minor / worth flagging for further review].

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*
Original (Week 5): "Model substantially outperforms baseline."
Rewritten: "The model showed a measured F1 improvement over the rule-based baseline
under a random split (0.598 vs 0.154). Under a client-grouped split, the observed
improvement was [insert number] — directionally still ahead of baseline, though this
is a more conservative estimate of true generalization."

Original: "ctr_b dominates feature importance."
Rewritten: "ctr_b was observed as the highest-weighted feature (~78% importance) in this
run. This is directional and specific to this dataset and time window, not a general
claim about CTR's causal role in content decline."

Original: "Model is useful as decision-support."
Rewritten: "Based on measured precision (0.70) and recall (0.52), the model appears
suitable as decision-support for prioritizing manual review — it should not be used as
an automated action trigger without further validation."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.